# Data Acquisition

This notebook loads `creditcard.csv` into a pandas dataframe, converts each row to a MongoDB document, and inserts all 284,807 records into the `transactions` collection in MongoDB Atlas.

**Imports and logging:**

In [ ]:
import pandas as pd
from pymongo import MongoClient
import os
import logging

# Configure logging to record all steps to a log file
logging.basicConfig(
    filename='data_acquisition.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

**Connect to MongoDB:**

In [ ]:
# Connect using a MongoDB URI supplied through the environment
try:
    uri = os.environ.get("MONGODB_URI")
    if not uri:
        raise ValueError("Set MONGODB_URI before running this notebook.")
    client = MongoClient(uri, serverSelectionTimeoutMS=10000)
    client.admin.command("ping")
    logging.info("Connected to MongoDB successfully")
    print("Connected successfully")
except Exception:
    logging.error("Could not connect to MongoDB; check your configuration.")
    raise


**Load CSV:**

In [ ]:
# Set CREDITCARD_CSV if the file is not in data/creditcard.csv.
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "pipeline" else Path.cwd()
CSV_PATH = Path(os.environ.get("CREDITCARD_CSV", str(PROJECT_ROOT / "data" / "creditcard.csv")))

try:
    df = pd.read_csv(CSV_PATH)
    if df.empty:
        raise ValueError("The input CSV is empty.")
    expected = {"Time", "Amount", "Class"} | {f"V{i}" for i in range(1, 29)}
    if set(df.columns) != expected:
        raise ValueError("The CSV columns do not match the expected credit card dataset.")
    logging.info(f"Loaded creditcard.csv with {len(df)} rows")
    print(f"Loaded {len(df)} rows")
except Exception:
    logging.error("Failed to load or validate the input CSV.")
    raise


**Insert into MongoDB:**

In [ ]:
# Load only into an empty collection; never delete existing data automatically.
try:
    db = client[os.environ.get("MONGODB_DATABASE", "ds4320_project2")]
    collection = db["transactions"]
    if collection.count_documents({}) != 0:
        raise RuntimeError(
            "The transactions collection is not empty. Use a new MONGODB_DATABASE "
            "for a fresh import, or skip acquisition and run the analysis notebook."
        )
    documents = df.to_dict(orient="records")
    collection.insert_many(documents)
    logging.info(f"Inserted {len(documents)} documents into transactions collection")
    print(f"Inserted {len(documents)} documents into transactions collection")
except Exception:
    logging.error("Data import stopped. Existing data was not automatically deleted.")
    raise


In [ ]:
# Verify documents were inserted correctly
try:
    count = collection.count_documents({})
    if count != len(df):
        raise ValueError(f"Expected {len(df)} documents, found {count}.")
    logging.info(f"Verified {count} documents in transactions collection")
    print(f"Verified: {count} documents in transactions collection")
except Exception as e:
    logging.error(f"Failed to verify insertion: {e}")
    raise